<a href="https://colab.research.google.com/github/saurav0530/SCGAuth/blob/main/sauravKumar_SCG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```
##############################################
#####                                    #####
#####    Name       : Saurav Kumar       #####
#####    Roll       : 1901EE54           ##### 
#####    Supervisor : Dr. Udit Satija    #####
#####                                    #####
##############################################
```



In [ ]:
#################################################
####           DEPENDENCIES-LOADING          ####
#################################################

In [ ]:
# Downloading the project folders from drive to local
# !pip install --upgrade --no-cache-dir gdown
!gdown --id 10mMjqlMIZ95Mr42U82FJsYlUpv5I4zXH
!unzip SCG.zip
!rm SCG.zip

In [ ]:
# Changing the directory
%cd SCG

In [ ]:
# For knowing the current working directory
%pwd

In [ ]:
# Installing all the dependencies
!pip3 install -r requirements.txt

In [ ]:
######################################################
####              PRE-PROCESSING(scg)             ####
######################################################

In [ ]:
# Downloading the dataset from drive to local
!pip install --upgrade --no-cache-dir gdown
!gdown --id 1KmDIw8vLH-b9ud4F25S3sMBt4hv_-YUR
!unzip scg.zip
!rm scg.zip
!mv files scg

In [ ]:
# Extracting raw signal from each SCG
# !python pre_processing/index.py

from typing import Mapping
import wfdb, os, shutil
import numpy as np
import pandas as pd
from os import listdir
import biosppy
import json, tqdm
import statistics
from scipy.fft import fft

dest_path = 'datasets/scg'
general_settings = 'settings/general_settings.json'
base_path = 'scg'

class NumpyEncoder(json.JSONEncoder):
    """ Special json encoder for numpy types """
    def default(self, obj):
        if isinstance(obj, (np.int_, np.intc, np.intp, np.int8, np.int16, np.int32, np.int64, np.uint8, np.uint16, np.uint32, np.uint64)):
            return int(obj)
        elif isinstance(obj, (np.float_, np.float16, np.float32, np.float64)):
            return float(obj)
        elif isinstance(obj, (np.ndarray,)):
            return obj.tolist()
        return json.JSONEncoder.default(self, obj)

this_dict = dict()

def make_segments(ecg):
    params = json.load(open(general_settings, 'r'))
    step = params['sampling']*params['slice_time']
    length = len(ecg)
    ans = []
    start=int(step/2)
    i=0
    while len(ans)<params['sample'] and start<length:
        ans.append(i)
        i = i+int(step/2)
        if i>=length:
            i = start
            start = start+int(step/2)
    return ans

def get_initial_csv(ecg_name, patient, file_path):  
    # ecg_name without extension    
    record = wfdb.rdsamp(base_path+'/'+ecg_name+patient, channels=[3])
    record = np.asarray(record[0])

    record = record[0:int(len(record)/5)*5]
    record = np.average(record.reshape(-1, 5), axis=1)
    record = record.reshape(len(record),1)

    record = (record - np.min(record))/np.ptp(record)
    
    df = pd.DataFrame(record)
    df.to_csv(file_path+'/preproc_csv/'+ecg_name+patient+'.csv', index=False, header=False)
    return make_segments(record)

extra_files = ["example.png","README.txt", "RECORDS", "RECORDS.txt", "SHA256SUMS.txt", "info.txt"]
temp = {f.split('.')[0] for f in os.listdir(base_path) if f not in extra_files}

if os.path.exists(dest_path):
    shutil.rmtree(dest_path)
    os.mkdir(dest_path)

if os.path.exists(dest_path+'/preproc_csv'):
    shutil.rmtree(dest_path+'/preproc_csv')

if not os.path.exists(dest_path+'/preproc_csv'):
    os.makedirs(dest_path+'/preproc_csv')

for f in tqdm.tqdm(temp):  
    if f=='m004' or f=='m018':
        continue  
    this_dict[f+'.csv'] = get_initial_csv(f[0], f[1:], dest_path)
    # break

with open(dest_path+'/start_points.json', 'w') as fp:
    json.dump(this_dict, fp, cls=NumpyEncoder)

In [ ]:
# Building the segments from each SCG signal
# !python pre_processing/make_segments.py

import numpy as np
import tqdm
import pandas as pd
import neurokit2 as nk
import os
import json

class NumpyEncoder(json.JSONEncoder):
    """ Special json encoder for numpy types """
    def default(self, obj):
        if isinstance(obj, (np.int_, np.intc, np.intp, np.int8, np.int16, np.int32, np.int64, np.uint8, np.uint16, np.uint32, np.uint64)):
            return int(obj)
        elif isinstance(obj, (np.float_, np.float16, np.float32, np.float64)):
            return float(obj)
        elif isinstance(obj, (np.ndarray,)):
            return obj.tolist()
        return json.JSONEncoder.default(self, obj)


with open("datasets/scg/start_points.json", 'r') as fid:
    dict_start_points = [json.loads(l) for l in fid][0]

base_path = "datasets/scg/preproc_csv"
base_path_segment = "datasets/scg"
general_settings = 'settings/general_settings.json'
params = json.load(open(general_settings, 'r'))
segment_length = params['slice_time']*params['sampling']

maps = {
    'b': '001',
    'm': '002',
    'p': '003'
}


for k, v in tqdm.tqdm(dict_start_points.items()):
    subjects = int(k.split('.')[0][1:])-1
    subjects = "{:03n}".format(subjects)
    samples = maps[k[0]]

    if not os.path.exists(base_path_segment + "/" + subjects):
        os.makedirs(base_path_segment + "/" + subjects)
    if not os.path.exists(base_path_segment + "/" + subjects + "/" + samples):
        os.makedirs(base_path_segment + "/" + subjects + "/" + samples)

    filename = base_path + "/" + k
    signal = pd.read_csv(filename, header=None)
    signal = signal.to_numpy()

    segments = []
    for r in v:
        record = signal[r:r+segment_length, :]
        # record = (record - np.min(record))/np.ptp(record)
        segments.append(record)
    
    segments = [s for s in segments if s.shape == (segment_length, 1)]

    for i, sn in enumerate(segments):
        result = sn
        df = pd.DataFrame(result)
        word = 'ss'
        df.to_csv(base_path_segment + "/" + subjects + "/" + samples + "/" + "{:03n}".format(i) + ".csv", index=False, header=False)

In [ ]:
# Creating train, test and validation dataset
# !python pre_processing/create_datasets.py

import os
import tqdm
import json
import random
from datetime import datetime

base_path = "datasets/scg/"


def get_list_of_class():
    classes = [name for name in os.listdir(base_path) if not os.path.isfile(os.path.join(base_path, name))]
    return [p for p in classes if not p == 'preproc_csv']

def create_ds(users, split_pct):
    assert(split_pct['train'] + split_pct['val'] + split_pct['test'] == 1)
    random.seed(datetime.now().timestamp())

    training   = []
    testing    = []
    validation = []

    for u in users:
        
        train   = []
        val     = []
        test    = []
        
        for i in os.listdir(base_path+u):
            signals = []
            for j in os.listdir(base_path+u+'/'+i):
                signals.append(u+'/'+i+'/'+j)
            random.shuffle(signals)

            for j in signals[:int(split_pct['train']*len(signals))]:
                train.append(j)

            for j in signals[int(split_pct['train']*len(signals)):int((split_pct['train']+split_pct['val'])*len(signals))]:
                val.append(j)

            for j in signals[int((split_pct['train']+split_pct['val'])*len(signals)):]:
                test.append(j)
        
        for s in train:
            training.append({"signal" : s, "code": u})

        for s in val:
            validation.append({"signal" : s, "code": u})

        for s in test:
            testing.append({"signal" : s, "code": u})

    random.shuffle(training)
    random.shuffle(testing)
    random.shuffle(validation)

    return training, validation, testing


def write_json(ds, fn):
    with open('datasets/' + fn, 'w') as fid:
        for tr in ds:
            json.dump(tr, fid)
            fid.write('\n')


def do_everything(split_pct, ds_filename):
    classes = get_list_of_class()
    train_ds, val_ds, test_ds = create_ds(classes, split_pct)
    print(train_ds)
    for ds, fn in zip([train_ds, val_ds, test_ds], [name + '_' + ds_filename for name in ['train', 'val', 'test']]):
        if len(ds) > 0:
            write_json(ds, fn)


classic_split = {'train': 0.7, 'val': 0.1, 'test': 0.2}
autoencoder_split = {'train': 0.8, 'val': 0.2, 'test': 0}
only_train_split = {'train': 1, 'val': 0, 'test': 0}
only_test_split = {'train': 0, 'val': 0, 'test': 1}

do_everything( autoencoder_split, 'autoencoder.json')
do_everything( classic_split, 'verification.json')
do_everything( classic_split, 'identification.json')

In [ ]:
#################################################
####               AUTO-ENCODER              ####
#################################################

In [ ]:
# For training autoencoders
!python autoencoder/train.py

In [ ]:
import os
import shutil
folder = 'saved/autoencoder/'
file = (os.listdir(folder)[1] if os.path.isdir(folder+os.listdir(folder)[1]) else os.listdir(folder)[0])
destt = 'saved/'+file+'_autoencoder'
if not os.path.exists(destt):
    os.makedirs(destt)

for f in os.listdir(folder+file+'/'):
    if f=='0.000-040-0.000.hdf5' or f=='preproc.bin' or f=='train_loss_metric.csv':
        shutil.copyfile(folder+file+'/'+f,destt+'/'+f)
shutil.rmtree(folder)

In [ ]:
# For plotting loss curves
import pandas as pd
import matplotlib.pyplot as plt

temp = pd.read_csv('/content/SCG/saved/1670125266-964_autoencoder/train_loss_metric.csv')
temp = temp.to_numpy()
plt.plot(temp[:, 0], temp[:, 1], color='r', label='train_loss')
plt.plot(temp[:, 0], temp[:, 2], color='g', label='val_loss')
plt.legend()

In [ ]:
# Temporary cell commands for deleting a folder if required
import shutil, os
folder = 'saved'
shutil.rmtree(folder)
os.mkdir(folder)

In [ ]:
######################################################
####                   TSNE PLOT                  ####
######################################################

In [ ]:
# For TSNE plot
import pandas as pd
import numpy as np
import os
import seaborn as sns
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.patheffects as PathEffects
from sklearn.manifold import TSNE

mod_path = '/content/SCG/saved/1670125266-964_autoencoder/0.000-040-0.000.hdf5'

def feature_extraction(model_path, output_layer, signals):
    import tensorflow_addons as tfa
    tfa.losses.TripletSemiHardLoss()

    dependencies = {'loss': tfa.losses.TripletSemiHardLoss()}
    model = tf.keras.models.load_model(model_path, custom_objects=dependencies)
    if not output_layer == "sequential":
        out = model.get_layer(output_layer).output
        model = tf.keras.Model(model.input, out)

    data_preproc = np.array([e for e in signals]).astype(float)
    features = model.predict(data_preproc)
    return features

def load_signal(record):
    signal = pd.read_csv(record, header=None)
    signal = signal.to_numpy()

    signal1 = np.asarray(signal)
    signal1 = signal1.reshape(10000)
    
    signal = np.expand_dims(signal, axis=-1)  # Create feature dimension
    return signal, signal1


sig = []
inp = []
tar = []

file_path = 'datasets/scg'
for i in os.listdir(file_path):
    if i == 'preproc_csv' or i == 'start_points.json':
        continue

    
  
    for j in os.listdir(file_path+'/'+i):
        for k in os.listdir(file_path+'/'+i+'/'+j):
            a, b = load_signal(file_path+'/'+i+'/'+j+'/'+k)
            sig.append(a)
            inp.append(b)
            tar.append(i)


# Utility function to visualize the outputs of t-SNE
plt.figure(figsize=(16, 10))

def fashion_scatter(x, colors, texts, plot):
    colors = np.asarray(colors)
    num_classes = len(np.unique(colors))
    classes = np.unique(colors)
    palette = np.array(sns.color_palette("hls", num_classes))
    print(palette.shape)
    ax = plt.subplot(1,2,plot, aspect='equal')
    sc = ax.scatter(x[:,0], x[:,1], lw=0, s=40, c=palette[colors.astype(int)])
    plt.xlim(-25, 25)
    plt.ylim(-25, 25)
    ax.axis('off')
    ax.axis('tight')
    
    txts = []
    z = {}
    for i in range(len(colors)):
        if i not in z:
            z[int(colors[i])] = []
        z[int(colors[i])].append(x[i])

    # for i in range(num_classes):
    #     xtext, ytext = np.median(z[i], axis=0)
    #     txt = ax.text(xtext, ytext, str(i), fontsize=14)
    #     txt.set_path_effects([
    #         PathEffects.Stroke(linewidth=5, foreground="w"),
    #         PathEffects.Normal()])
    #     txts.append(txt)

    plt.axis('off')
    plt.title(texts)

fashion_tsne_inp = TSNE(random_state=1).fit_transform(inp)
fashion_scatter(fashion_tsne_inp, tar, "Input segments", 1)

X = feature_extraction(mod_path, 'flatten', sig)
fashion_tsne_X = TSNE(random_state=1).fit_transform(X)
fashion_scatter(fashion_tsne_X, tar, "Encoded segments", 2)

In [ ]:
#################################################
####               VERIFICATION              ####
#################################################

In [ ]:
# For training Siamese network
!python verification/train.py

In [ ]:

import os
import shutil
folder = 'saved/verification/'
file = (os.listdir(folder)[1] if os.path.isdir(folder+os.listdir(folder)[1]) else os.listdir(folder)[0])
destt = 'saved/'+file+'_verification'
if not os.path.exists(destt):
    os.makedirs(destt)

for f in os.listdir(folder+file+'/'):
    if f=='0.114-0.954-047-0.044-0.983.hdf5' or f=='preproc.bin' or f=='train_loss_metric.csv':
        shutil.copyfile(folder+file+'/'+f,destt+'/'+f)
shutil.rmtree(folder)

In [ ]:
# For plotting loss curves
import pandas as pd
import matplotlib.pyplot as plt

temp = pd.read_csv('/content/BTP/saved/1668593916-226_verification/train_loss_metric.csv')
temp = temp.to_numpy()
plt.title("Verification")
plt.plot(temp[:, 0], temp[:, 1], color='r', label='train_loss')
plt.plot(temp[:, 0], temp[:, 2], color='g', label='val_loss')
plt.title("Verification")
plt.legend()
plt.show()

In [ ]:
!python verification/predict.py --model_path "/content/SCG/saved/verification/1670126673-928/0.348-0.854-017-0.207-0.926.hdf5"

In [ ]:
#################################################
####              IDENTIFICATION             ####
#################################################

In [ ]:
# For training Siamese identification network
!python identification/train.py

In [ ]:
# 1641449739-387_1L-Siamese 0.461-0.836-019-0.349-0.856.hdf5
import os
import shutil
folder = 'saved/identification/'
file = (os.listdir(folder)[1] if os.path.isdir(folder+os.listdir(folder)[1]) else os.listdir(folder)[0])
destt = 'saved/'+file+'_identification'
if not os.path.exists(destt):
    os.makedirs(destt)

for f in os.listdir(folder+file+'/'):
    if f=='0.040-1.000-027-0.075-0.977.hdf5' or f=='preproc.bin' or f=='train_loss_metric.csv':
        shutil.copyfile(folder+file+'/'+f,destt+'/'+f)
shutil.rmtree(folder)

In [ ]:
# For plotting loss curves
import pandas as pd
import matplotlib.pyplot as plt

temp = pd.read_csv('/content/BTP/saved/identification/1668594439-870/train_loss_metric.csv')
temp = temp.to_numpy()

plt.plot(temp[:, 0], temp[:, 1], color='r', label='train_loss')
plt.plot(temp[:, 0], temp[:, 2], color='g', label='val_loss')
plt.title("Identification")
plt.legend()
plt.show()

In [ ]:
!python identification/predict.py --model_path "/content/SCG/saved/identification/1670126888-742/0.337-0.887-050-0.283-0.920.hdf5"

In [ ]:
# Temporary cell commands for deleting a folder if required
import shutil, os
folder = 'SCG'
shutil.rmtree(folder)

In [ ]:
#################################################
####             SAVING TO GDRIVE            ####
#################################################

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# %cd SCG
%pwd

In [ ]:
# Updating the changed folder to drive
%cd ..
!zip -r SCG.zip SCG
%cp SCG.zip drive/MyDrive/Saurav/SCG.zip
!rm SCG.zip

In [ ]:
#################################################
####             IMPORTANT LINKS             ####
#################################################

# 1. https://www.datacamp.com/tutorial/introduction-t-sne